# Fase 3 — Paso 3.2: Módulo No Supervisado (Clustering)

K-Means sobre variables sociodemográficas y geográficas. Validación con Método del Codo y Coeficiente de Silueta.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('../../data/processed/casos_limpios.csv')

# Ajustar a las columnas reales del dataset
vars_numericas = ['edad']
vars_categoricas = ['sexo', 'provincia']

X = df[vars_numericas + vars_categoricas].dropna()

In [ ]:
preprocesador = ColumnTransformer([
    ('num', StandardScaler(), vars_numericas),
    ('cat', OneHotEncoder(handle_unknown='ignore'), vars_categoricas)
])

X_transformado = preprocesador.fit_transform(X)

## Método del Codo

In [ ]:
inercias = []
rango_k = range(2, 11)

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_transformado)
    inercias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(rango_k), inercias, marker='o')
plt.xlabel('Número de clústeres (k)')
plt.ylabel('Inercia')
plt.title('Método del Codo')
plt.show()

## Coeficiente de Silueta

In [ ]:
siluetas = []

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    etiquetas = km.fit_predict(X_transformado)
    score = silhouette_score(X_transformado, etiquetas)
    siluetas.append(score)

plt.figure(figsize=(8, 5))
plt.plot(list(rango_k), siluetas, marker='o', color='orange')
plt.xlabel('Número de clústeres (k)')
plt.ylabel('Coeficiente de Silueta')
plt.title('Validación por Coeficiente de Silueta')
plt.show()

In [ ]:
# Definir K_OPTIMO según los gráficos anteriores
K_OPTIMO = 4

modelo_kmeans = KMeans(n_clusters=K_OPTIMO, random_state=RANDOM_STATE, n_init=10)
df.loc[X.index, 'cluster'] = modelo_kmeans.fit_predict(X_transformado)

df['cluster'].value_counts()